### Final model

Goal: to train the best-performing model on the full training dataset using the best hyperparameters found during tuning.

Best model from notebook 02: LGBM with Optuna-tuned hyperparameters (PR-AUC: 0.702)

The final model will be saved with `joblib` as:
`models/final_model.pkl`

This `.pkl` file stores the trained model so it can be reused later without retraining.


After training, we will perform:

- Threshold tuning to choose the best classification threshold
- SHAP Global analysis to understand overall feature importance
- SHAP Local analysis to explain individual predictions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, precision_recall_curve

from imblearn.pipeline import Pipeline

from lightgbm import LGBMClassifier

import joblib
import shap

In [ ]:
df = pd.read_csv("../../data/Customer-Churn-Records.csv")

In [ ]:
X = df.drop(columns=["Complain", "Exited", "RowNumber", "CustomerId", "Surname"])
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
categorical = X.select_dtypes(include='object').columns
numerical = X.select_dtypes(exclude='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        ('scaler', StandardScaler(), numerical)
    ],
    remainder='passthrough'
)

# Best params from Optuna (50 trials), PR-AUC: 0.702 (CV on X_train)
best_params = {
    'n_estimators': 630,
    'max_depth': 4,
    'learning_rate': 0.0168,
    'subsample': 0.735,
    'colsample_bytree': 0.636,
    'reg_alpha': 2.25e-05,
    'reg_lambda': 8.07e-08
}

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(**best_params, scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1))
])

final_model.fit(X_train, y_train)

In [ ]:
joblib.dump(final_model, '../../models/final_model.pkl')

In [ ]:
y_prob = final_model.predict_proba(X_test)[:, 1]

average_precision_score(y_test, y_prob)

We trained the best-performing LightGBM model on the full training dataset using the optimal hyperparameters found during Optuna tuning. The model was then evaluated on the unseen test set and achieved:

**PR-AUC: 0.7233**

This result is slightly higher than the mean cross-validation PR-AUC obtained during model selection (**0.702**), indicating that the model is not suffering from overfitting.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1])
best_threshold = thresholds[np.argmax(f1_scores)]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

candidate_thresholds = [0.30, 0.40, 0.50, best_threshold, 0.70]

threshold_results = []

for threshold in candidate_thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 3),
        "precision": round(precision_score(y_test, y_pred_threshold), 3),
        "recall": round(recall_score(y_test, y_pred_threshold), 3),
        "f1_score": round(f1_score(y_test, y_pred_threshold), 3)
    })

threshold_results_df = pd.DataFrame(threshold_results)
threshold_results_df

## Threshold Tuning

I selected a threshold of 0.50 because it provides a better balance between precision and recall than 0.40. Lowering the threshold from 0.50 to 0.40 increases recall by only 5.9 percentage points (77.9% → 83.8%), while precision drops by 9.0 percentage points (52.2% → 43.2%)

Although the threshold of 0.592 achieved the highest F1-score, maximizing F1 was not the main objective. In this business scenario, recall is more important because missing a customer who is likely to churn is more costly than contacting a customer who would have stayed anyway

Business-wise, this means the model correctly identifies approximately 78% of customers who are likely to churn, while about 52% of the customers flagged by the model are actual churners.

In [ ]:
lgbm_model = final_model.named_steps["classifier"]
X_test_processed = final_model.named_steps["preprocessor"].transform(X_test)

explainer = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_test_processed)

In [ ]:
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
feature_names_clean = [f.split('__')[1] for f in feature_names]

shap.summary_plot(shap_values, X_test_processed, feature_names=feature_names_clean, show=False)
plt.tight_layout()
plt.savefig('../../reports/figures/shap_global.png', dpi=150, bbox_inches='tight')
plt.show()

### SHAP Global Analysis

The SHAP beeswarm plot shows both the importance of each feature and how feature values influence the model's predictions.

1. **NumOfProducts** was identified as the most influential feature. High values of this feature often increase churn risk, but some high values also decrease it. This suggests a non-linear relationship between the number of products and churn.<br>
**From a business perspective**, customers with higher product portfolios should be monitored more closely. The bank could investigate why customers with certain numbers of products are more likely to leave and design targeted retention campaigns for these segments.

2. **Age** was the second most important feature. Older customers (red points) are mostly located on the positive side of the plot, indicating a higher probability of churn.<br>
**From a business perspective**, older customers may benefit from dedicated retention programs or personalized communication aimed at reducing churn risk.

3. **IsActiveMember** was another highly important feature. Active customers generally decrease churn risk, while inactive customers increase it. <br>
**From a business perspective**, increasing customer engagement should be a priority.

In [ ]:
results = pd.DataFrame({
    "y_true": y_test.reset_index(drop=True),
    "y_prob": y_prob
})

# churn 100%
results[results["y_true"] == 1].sort_values("y_prob", ascending=False).head()

In [ ]:
idx = 614

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_test_processed[idx],
        feature_names=feature_names_clean
    )
)

In [ ]:
# border (probability 0.5)
results["dist"] = (results["y_prob"] - 0.5).abs()
results.sort_values("dist").head()

In [ ]:
idx = 1828

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_test_processed[idx],
        feature_names=feature_names_clean
    )
)

### SHAP Local Analysis

**Confident churner (id 614)**

The model classifies this customer as churn with high confidence (f(x) = 6.17, probability 0.998) because almost every feature points the same way. NumOfProducts is the dominant driver (+3.21), which matches Bakdaulet's EDA finding that customers with 3 to 4 products churn at 83 to 100 percent. Age (+1.99) and inactivity (+0.63) reinforce this, exactly the factors that scored strongest in the hypothesis tests (Age, Cohen's d 0.74). Germany, female gender and high balance add smaller but consistent signals. CreditScore is the only factor that slightly reduces churn, but its impact is very small.

**Borderline customer (id 1828)**

The model is undecided about this customer (f(x) close to 0, probability near 0.5) because the features pull in opposite directions. Young age is the strongest protective force (-0.99), pushing toward staying, while a low number of products (+0.82), inactivity (+0.41) and female gender (+0.24) push toward churn. These effects roughly cancel out, which is why the prediction lands on the decision boundary. Note that here a low product count drives churn, whereas for the confident churner a high count did, capturing the non linear product effect also seen in the EDA